# 🧬 Single-Cell Analysis of Fly Ovary Cells

**Project Goal:**  
Analyze a heterogeneous mix of fly cells from different tissues to identify **cell types**, **marker genes**, and **likely tissue origins**.

---

## Dataset
- File: `data/ovary.tsv`  
- Type: single-cell RNA-seq counts  
- Format: rows = genes, columns = cells (check orientation)  

---

## Tools
- **Python**  
- **Scanpy** (for preprocessing, clustering, visualization)  
- **Pandas** (for data handling)  
- **Matplotlib / Seaborn** (for plots)  
- **FlyBase or literature** (to identify cell-types from marker genes)  

---

## Analysis Plan

1. **Data Loading & QC**
   - Load TSV into a Scanpy `AnnData` object.
   - Inspect cell and gene counts.
   - Filter low-quality cells and genes.

2. **Normalization & Preprocessing**
   - Normalize counts per cell.
   - Log-transform the data.
   - Identify highly variable genes.
   - Scale and run PCA.

3. **Dimensionality Reduction**
   - Compute neighborhood graph.
   - Run **UMAP** and/or **t-SNE** for visualization.

4. **Clustering**
   - Perform clustering (e.g., Leiden algorithm).
   - Visualize clusters on UMAP/t-SNE.

5. **Marker Gene Identification**
   - Perform **differential expression analysis** for each cluster.
   - Identify **marker genes** that define cell-types.

6. **Cell-Type Annotation**
   - Use marker genes and resources like **FlyBase** to infer likely cell-types.
   - Summarize which tissues are present and how many cell-types were found.

---

## Expected Results
- Number of clusters / cell-types.  
- Marker genes per cell-type.  
- Tentative annotation of cell-types with tissue origin.  

---

## Notes
- Ensure **gene names are unique** to avoid errors.  
- Save processed data to `write/ovary.h5ad` for future analysis.

## PARTIE 1: Settings ##

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import numpy as np
import scipy.sparse as sp

**Trouver path:**

In [ ]:
import os
 
print("Current directory:")
print(os.getcwd())

print("\nFiles here:")
print(os.listdir())

print("\nSearching for ovary.tsv ...")
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".tsv"):
            print(os.path.join(root, file))

### a) Load the data

In [ ]:
# Read just the first few rows to understand the file structure
df_peek = pd.read_csv("./data/ovary.tsv", sep="\t", nrows=5)
print("Shape:", df_peek.shape)
print("First column name:", df_peek.columns[0])
print("Columns 0-5:", list(df_peek.columns[:5]))
print("Index:", df_peek.index.tolist())
print(df_peek.iloc[:3, :3])  # top-left corner of the file

In [ ]:
# Load TSV with gene names as index
#df = pd.read_csv("ovary.tsv", sep="\t", index_col=0)  # Path on Clotilde computer
df = pd.read_csv("./data/ovary.tsv", sep="\t", index_col=0)

# Verify BEFORE any conversion
print("Gene names sample:", df.index[:10].tolist())
print("Shape (genes x cells):", df.shape)

# Convert data values to numeric (index is already string gene names, safe)
df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

# Create AnnData
adata = sc.AnnData(df.T)  # transpose: cells x genes

print("adata.var_names (genes):", adata.var_names[:10].tolist())
print("adata.obs_names (cells):", adata.obs_names[:5].tolist())

adata.var_names_make_unique()  # NOTE: call it as a function with ()

# Filter
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)

# Check markers are findable
markers = ['nos', 'bam', 'tj', 'eya', 'fas3']
for gene in markers:
    matches = [g for g in adata.var_names if gene.lower() in g.lower()]
    print(f"{gene} -> {matches[:3]}")

### b) Setting and logging ###

In [ ]:
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.set_figure_params(dpi=80, facecolor="white")
sc.logging.print_header()

results_file = "write/ovary.h5ad"  # the file that will store the analysis results

## PARTIE 2: Pre-processing ##

### a) Quality control: 

In [ ]:
import scipy.sparse as sp

# Step 1: Compute QC metrics
if sp.issparse(adata.X):
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
else:
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1)
    adata.obs['n_counts'] = adata.X.sum(axis=1)

# Step 2: Calculate percent mitochondrial genes
mito_genes = [g for g in adata.var_names if g.startswith('mt-') or g.startswith('MT-')]
if sp.issparse(adata.X):
    adata.obs['percent_mito'] = (adata[:, mito_genes].X.sum(axis=1).A1 / adata.obs['n_counts']) * 100
else:
    adata.obs['percent_mito'] = (adata[:, mito_genes].X.sum(axis=1) / adata.obs['n_counts']) * 100

# Step 3: Visualize QC metrics
sc.pl.violin(adata, ['n_genes', 'n_counts', 'percent_mito'], jitter=0.4, multi_panel=True)

# Scatter plots similar to Seurat's FeatureScatter
sc.pl.scatter(adata, x='n_counts', y='percent_mito', color='percent_mito')
sc.pl.scatter(adata, x='n_counts', y='n_genes', color='n_genes')

# Step 4: Filter cells based on thresholds
adata = adata[
    (adata.obs['n_genes'] > 200) &
    (adata.obs['n_genes'] < 5000) &
    (adata.obs['percent_mito'] < 10),
    :
]

After creating the AnnData object (Scanpy equivalent of a Seurat object), the next step is to assess the quality of the cells in the dataset. The goal is to **remove low-quality or abnormal cells** that could bias downstream analysis.  

### Metrics Calculated

1. **Number of genes per cell (`n_genes`)**  
   - Counts how many genes are detected in each cell.  
   - Cells with very few detected genes may be dead or poorly sequenced.  
   - Cells with unusually many genes may represent **doublets** (two cells captured as one).  

2. **Total counts per cell (`n_counts`)**  
   - Total number of transcripts detected in each cell.  
   - Helps identify cells with abnormally high or low sequencing depth.  

3. **Percent mitochondrial transcripts (`percent_mito`)**  
   - Fraction of a cell's transcripts originating from mitochondrial genes.  
   - High mitochondrial content often indicates stressed or dying cells.  
   - For fly ovary data, mitochondrial genes are typically prefixed with `mt-` or `MT-`.  

### Visualization

**Violin plots** are used to examine the distribution of these metrics across all cells. This allows us to **identify outliers** rather than applying arbitrary cutoffs.  

### Filtering
After inspecting the distributions, cells with extreme values are removed:
  - Cells with too few or too many genes.  
  - Cells with excessively high mitochondrial transcript percentages.  
This step ensures that downstream analyses, like clustering and differential expression, focus on **high-quality, biologically meaningful cells**.  

### b) Normalization:


After quality control, the next step is **normalization**.  

In scRNA-seq, the amount of RNA captured per cell can vary greatly. To make gene expression values comparable across cells, we normalize each cell's counts:

1. Scale each cell's total counts to a constant (default: 10,000).  
2. Log-transform the scaled counts to reduce skewness and make the data more suitable for downstream analysis.

This is analogous to TPM (Transcripts Per Million) normalization used in bulk RNA-seq.


In [ ]:
adata.raw = adata.copy()
# Normalize the data (total-count normalize to 10,000 counts per cell and log-transform)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)  # log(x + 1) transform

### Identify highly variable gene :

In [ ]:
adata.var.columns

In [ ]:
# Recompute HVGs cleanly
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor='seurat'
)

# Make base plot
sc.pl.highly_variable_genes(adata, log=True, show=False)

# Get top 20 by normalized dispersion
hvg = adata.var[adata.var['highly_variable']]
top_features = hvg.sort_values(
    'dispersions_norm',
    ascending=False
).head(20).index

# Add labels
for gene in top_features:
    x = adata.var.loc[gene, 'means']
    y = adata.var.loc[gene, 'dispersions_norm']
    plt.text(x, y, gene, fontsize=8)

plt.show()

### Scaling

In [ ]:
sc.pp.scale(adata, max_value=10)

### Linear Dimensionality reduction with PCA

In [ ]:
sc.tl.pca(adata, svd_solver='arpack') 

variance_ratio = adata.uns['pca']['variance_ratio']

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(variance_ratio) + 1), variance_ratio, 'o-', markersize=4, color='black')
ax.axvline(x=20, color='red', linestyle='--', label='Cutoff : PC 20')
ax.set_yscale('log')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (log scale)')
ax.set_title('Elbow Plot - PCA Variance Ratio')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation** 

Steep drop at the beginning (PCs 1–5): These PCs capture the strongest sources of biological variation in your data, likely major cell type differences.  
Gradual flattening (PCs 5–20): These still capture real but subtler biological signals, such as cell subtypes or states.  
Flat tail (PCs ~20+): These mostly represent technical noise or stochastic cell-to-cell variation, not informative. 

For the heatmap we decide to only keep the first 20 principal analysis component

In [ ]:
def plot_pc_heatmap(adata, n_pcs=20, n_genes=10, n_cells=500):
    loadings = adata.varm['PCs']
    embeddings = adata.obsm['X_pca']
    
    ncols = 4
    nrows = int(np.ceil(n_pcs / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
    axes = axes.flatten()
    
    for pc_idx in range(n_pcs):
        ax = axes[pc_idx]
        pc_loadings = loadings[:, pc_idx]
        
        # Balanced top/bottom genes
        top_pos = np.argsort(pc_loadings)[-(n_genes):]
        top_neg = np.argsort(pc_loadings)[:n_genes]
        gene_idx = np.concatenate([top_neg, top_pos])
        gene_names = adata.var_names[gene_idx]
        
        # Sort cells by PC score
        cell_order = np.argsort(embeddings[:, pc_idx])
        step = max(1, len(cell_order) // n_cells)
        cell_subset = cell_order[::step][:n_cells]
        
        # Extract expression
        if hasattr(adata.X, 'toarray'):
            expr = adata.X[cell_subset][:, gene_idx].toarray()
        else:
            expr = adata.X[cell_subset][:, gene_idx]
        
        # Plot
        im = ax.imshow(expr.T, aspect='auto', cmap='RdBu_r',
                       vmin=-2, vmax=2, interpolation='nearest')
        
        ax.set_title(f'PC{pc_idx + 1}', fontsize=11, fontweight='bold')
        ax.set_yticks(range(len(gene_names)))
        ax.set_yticklabels(gene_names, fontsize=7)
        ax.set_xticks([])  # hide cell numbers - not informative
        
        # Divider line between neg and pos genes
        ax.axhline(y=n_genes - 0.5, color='white', linewidth=1.5, linestyle='--')
        
        plt.colorbar(im, ax=ax, shrink=0.6, label='Scaled expr')
    
    # Hide unused subplots
    for i in range(n_pcs, len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle('PC Heatmaps — Top Driving Genes', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('pc_heatmap_clean.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_pc_heatmap(adata, n_pcs=20, n_genes=10, n_cells=500)

## Non-linear dimensionality reduction

In [ ]:
print(adata.obs.columns)


In [ ]:
# Compute neighborhood graph (required before UMAP/tSNE)
# n_pcs match our elbow plot decision : 20
sc.pp.neighbors(adata, n_pcs=20)
sc.tl.leiden(adata, resolution=0.5)  # adjust resolution to get more/fewer clusters

# Step 2: Run UMAP
sc.tl.umap(adata)

# Run t-SNE
sc.tl.tsne(adata, n_pcs=20)

# Visualize side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata, ax=axes[0], color ='leiden', show=False, title='UMAP')
sc.pl.tsne(adata, ax=axes[1], color ='leiden', show=False, title='tSNE')
plt.tight_layout()
plt.show()

#### UMAP (Uniform Manifold Approximation and Projection)
Preserves both local and global structure of the data. Better at
retaining **trajectory-like/continuous structures**, making it preferred
for differentiation datasets.

#### t-SNE (t-distributed Stochastic Neighbor Embedding)
Prioritizes local neighborhood structure. Produces **well-separated
distinct clusters**, making it visually striking when cell populations
are discrete.

We color by Leiden. Leiden is a clustering algorith that was already run on our data. It groups celll into communities based on their similarity in the neighbor graph. 

We can see that the cell are clustered into groups but we have a problem : What's the meaning of the cluster ?  

To answer this questions we plor Drosohila ovary markers like nos, bam, tj, eya, fas3 onto the UMA to asssigh identities to each cluster. 

nos → marks germline stem cells / early germline  
bam → marks cystoblasts / differentiating germline  
tj → marks escort cells and follicle stem cells  
eya — marks follicle cells  
fas3 → marks follicle cells / polar cells  


In [ ]:
# Plot markers on UMAP to identify clusters
markers = ['nos', 'bam', 'tj', 'eya', 'Fas3']  # Note: Fas3 is capitalized in Drosophila

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Plot leiden clusters first, then each marker
sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, title='Leiden clusters')
for i, gene in enumerate(markers):
    sc.pl.umap(adata, color=gene, ax=axes[i+1], show=False, 
               title=gene, color_map='YlOrRd', vmax='p99')

plt.tight_layout()
plt.show()

In [ ]:
# After visually inspecting the plots above, assign cell type labels
# Edit this dictionary to match what you observe in YOUR data
cell_type_labels = {
    '0':  'follicle cells',       # high eya / Fas3
    '1':  'germline stem cells',  # high nos
    '2':  'escort cells',         # high tj
    '3':  'cystoblasts',          # high bam
    # ... fill in remaining clusters 4-13
}

adata.obs['cell_type'] = adata.obs['leiden'].map(cell_type_labels)

# Plot with cell type labels
sc.pl.umap(adata, color='cell_type', title='Cell types')

This look promising ¨!  
I will redo the same thing with more marker and see if I can get a finer refinment of my cell type, 

In [ ]:
# Extended marker panel for Drosophila ovary cell types
markers_extended = {
    # Germline
    'nos':   'germline stem cells / early germline',
    'bam':   'cystoblasts / differentiating germline',
    'orb':   'developing cysts (egg chambers)',
    'spn-E': 'nurse cells / germline',
    'stau':  'oocyte / nurse cells',
    'grk':   'oocyte (grk is oocyte-specific)',
    
    # Somatic niche
    'tj':    'escort cells + follicle stem cells',
    'eya':   'follicle cells',
    'Fas3':  'follicle cells / polar cells',
    'cas':   'terminal filament + cap cells',
    'hh':    'cap cells (niche)',
    'piwi':  'cap cells + GSCs',
    
    # Specific follicle subtypes
    'br':    'follicle cells (late)',
    'slbo':  'border cells',
    'upd1':  'cap cells (JAK-STAT ligand)',
    
    # Muscle / other
    'how':   'muscle / somatic',
    'vkg':   'hemocytes / ECM',
}

# Plot all on UMAP
genes_to_plot = list(markers_extended.keys())

n = len(genes_to_plot)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 4))
axes = axes.flatten()

for i, gene in enumerate(genes_to_plot):
    if gene in adata.var_names:
        sc.pl.umap(adata, color=gene, ax=axes[i], show=False,
                   title=f"{gene}\n({markers_extended[gene]})",
                   color_map='YlOrRd', vmax='p99')
    else:
        axes[i].set_title(f"{gene} - NOT FOUND")
        axes[i].axis('off')

# Hide unused axes
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

Based on result here are my cluster assignment : 

In [ ]:
cell_type_labels = {
    '0':  'escort cells',
    '1':  'escort cells',
    '2':  'follicle cells',
    '3':  'follicle cells',
    '4':  'follicle cells',
    '5':  'muscle / somatic',
    '6':  'late germline / nurse cells',
    '7':  'germline cysts',
    '8':  'late germline / nurse cells',
    '9':  'early germline / cystoblasts',
    '10': 'GSC / early germline',
    '11': 'germline transition',
    '12': 'follicle cells',
    '13': 'escort cells',
    '14': 'germline cysts',
    '15': 'escort cells',
    '16': 'follicle cells',
    '17': 'follicle cells',
    '18': 'follicle cells',
}

adata.obs['cell_type'] = adata.obs['leiden'].map(cell_type_labels)

sc.pl.umap(adata, color='cell_type',
           legend_loc='on data',
           legend_fontsize=8,
           title='Cell type annotations',
           palette='tab20',
           show=False)

plt.tight_layout()
plt.savefig('cell_types_umap.png', dpi=300, bbox_inches='tight')
plt.show()